In [ ]:
import pandas as pd
import requests
import json


#%% md

Resources of how to obtain data on GitHub
  * [GitHub API](https://docs.github.com/en/rest)
  * **Rerunning.** Run All is the normal way to refresh the results. Rerunning one collection cell (the pull request cell or the issue cell) first clears the results it produces, plus the joined table and the plots' inputs. A failed rerun therefore cannot leave older numbers in place. After a successful rerun, run the DataFrame, join, and plotting cells again.
  * Clearing variables does not remove output already rendered on this page. Tables and plots shown before a failed rerun stay visible until their cells run again.
  * Set the `GITHUB_TOKEN` environment variable before starting Jupyter. The notebook reads it from the environment and never accepts a token in a cell.
  * [How to create GitHub token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/creating-a-personal-access-token#creating-a-token) or you could use this [Generate Token](https://github.com/settings/tokens/new?scopes=repo) if you do not want to read  details.
  * Activity is keyed by GitHub login, so public profile names are not needed.


In [ ]:
# Sample repository. Replace it with your own repository
reponame = ""
# Date to start collecting data from
ignore_date = pd.to_datetime("2023-02-16T00:00:00-00:00", utc=True)
# File extensions to ignore from the graphs
ignored_file_extensions = [
    ".pdf",
    ".xml",
    ".html",
    ".css",
    ".log",
    ".txt",
    ".json",
    ".js",
]


In [ ]:
# The notebook uses the collector's request helper, so timeouts, HTTP errors, and invalid JSON
# are handled in one place. Run the notebook with src/ as its working directory.
import os
import sys

sys.path.insert(0, os.getcwd())
import CollectFiles
from CollectFiles import GitHubError, github_request, require_token


# GitHub returns at most 3000 changed files for one pull request
PR_FILES_LIMIT = 3000


def fetch_page(url):
    # Checks the token, then returns the parsed JSON for one request and the URL of the next page (None on the last page).
    # Raises GitHubError on any failure; never returns None.
    require_token()
    return github_request(url)


def fetch(url):
    # Returns the parsed JSON for one request
    data, _ = fetch_page(url)
    return data


def fetch_all(url, limit=None):
    # Follows the exact next-page URL that GitHub returns until there is no next page, and returns every item.
    # Raises GitHubError if any page fails, so a partial list is never returned.
    # If limit is given and the items reach it, raises GitHubError, even on the last page:
    # GitHub may have cut the list short, so completeness cannot be established.
    first_url = url
    items = []
    while url:
        data, url = fetch_page(url)
        items.extend(data)
        if limit is not None and len(items) >= limit:
            raise GitHubError(
                "{} entries reached the limit of {} for {}; the complete list cannot be established".format(
                    len(items), limit, first_url)
            )
    return items


In [ ]:
# Unattributed-account counts for each collection step
unattributed = {"pr": 0, "issues": 0}

# Results built from every step. Any step that is rerun makes these stale.
JOINED_RESULTS = ["df_list", "df_merged"]


def invalidate(names, step):
    # Each collection step calls this first, before any request can fail, so a failed rerun
    # cannot leave older results or DataFrames behind. It also clears the joined table and plots' inputs.
    for name in list(names) + JOINED_RESULTS:
        globals().pop(name, None)
    unattributed[step] = 0


In [ ]:
# Activity is keyed by GitHub login, which every PR, review, issue, and comment carries.
# Public profile names are not needed, so users with the same first name stay separate.

def login_of(user_obj, step):
    # GitHub returns "user": null for deleted accounts, so there is no login to credit
    if user_obj is None:
        unattributed[step] += 1
        return None
    return user_obj["login"]


In [ ]:
# This function counts pull request activity, keyed by GitHub login.
# Population: the closed PRs merged after ignore_date (their authors, files, and reviewers).


def pullrequest_details(reponame):
    invalidate(
        [
            "contributor_pr_count",
            "contributor_pr_review_count",
            "contributor_changedFiles_count",
            "contributor_changedLOC",
            "df_pr",
            "df_prReviews",
            "df_pr_ChangedFiles",
            "df_pr_ChangedLOC",
        ],
        "pr",
    )
    contributor_pr_count = dict()
    contributor_pr_review_count = dict()
    contributor_changedFiles_count = dict()
    contributor_changedLOC = dict()

    # loop though all the commit pages until the last returned empty page
    ipage = 1
    while True:
        spage = str(ipage)
        pr_api = "https://api.github.com/repos/" + reponame + "/pulls?page=" + spage + "&per_page=100&state=closed"
        pr_list = fetch(pr_api)

        # break out of the while loop if there are no more commits in the pages
        if len(pr_list) == 0:
            break

        # iterate through the list of pull requests in page
        for pr_obj in pr_list:
            pr_number = pr_obj["number"]

            # Only count PRs that were merged
            if pr_obj["merged_at"] is not None:
                merged_at = pr_obj["merged_at"]

                merged_time = pd.to_datetime(merged_at, utc=True)

                # Only count PRs that were merged strictly after the ignore_date
                if merged_time > ignore_date:
                    author = login_of(pr_obj["user"], "pr")

                    # A deleted author's metrics cannot be credited, but reviewers of their PR still are
                    if author is not None:
                        # Increase contributor's PR count
                        contributor_pr_count[author] = contributor_pr_count.get(author, 0) + 1

                        # Print the title of the PR (for debugging)
                        # print(pr_obj['title'])

                        """
                            We want to identify the authors of each chached file in the PR. First, we 
                            shall collect a list of PR-files and list of PR-commits. Second, we iterate
                            through a list of commits and pick the files changed in each commit and identify
                            the commit author as well as the files they changed. Lastly, we compare the files 
                            in the commits and those in PR-files and assign them to the different authors. 
                            """

                        # we only care about changed files in a PR, not the individual commits
                        pr_changedFiles_api = "https://api.github.com/repos/" + reponame + "/pulls/" + str(pr_number) + "/files?per_page=100"
                        pr_changedFiles_list = fetch_all(pr_changedFiles_api, limit=PR_FILES_LIMIT)

                        # Add the number of changed files to the contributor's count and lines of code changed
                        for file_obj1 in pr_changedFiles_list:
                            file = file_obj1["filename"]

                            # if file does not end with any of the ignored file extensions (declared above),
                            # add it to the LOC count & changed files count
                            if not any(file.endswith(x) for x in ignored_file_extensions):

                                # A note about the changed files, it does not track unique files,
                                # but rather the number of files changed in a PR irregardless if they already changed them before or not.
                                contributor_changedFiles_count[author] = contributor_changedFiles_count.get(author, 0) + 1

                                contributor_changedLOC[author] = contributor_changedLOC.get(author, 0) + file_obj1["changes"]

                                # Optional print statement to see the files and LOC changed by each contributor
                                # print(author + " " + str(file) + " " + str(file_obj1['changes']) + " " + str( contributor_changedLOC.get(author, 0)))

                    pr_reviews_api = "https://api.github.com/repos/" + reponame + "/pulls/" + str(pr_number) + "/reviews?per_page=100"

                    pr_reviews_list = fetch_all(pr_reviews_api)

                    # Check if we have already counted a reviewer
                    # This is to ensure that people that just commented on a PR receive proper credit
                    # Even if they have not approved the PR.
                    already_reviewed = set()
                    for pr_review_obj in pr_reviews_list:
                        reviewer = login_of(pr_review_obj["user"], "pr")
                        # Skip deleted accounts, and never credit an author for reviewing their own PR
                        if reviewer is None or reviewer == author:
                            continue
                        if reviewer not in already_reviewed:
                            already_reviewed.add(reviewer)
                            # print("Reviewed By: " + reviewer)
                            contributor_pr_review_count[reviewer] = contributor_pr_review_count.get(reviewer, 0) + 1

        ipage += 1
    return (
        contributor_pr_count,
        contributor_pr_review_count,
        contributor_changedFiles_count,
        contributor_changedLOC,
    )


(
    contributor_pr_count,
    contributor_pr_review_count,
    contributor_changedFiles_count,
    contributor_changedLOC,
) = pullrequest_details(reponame)

print("contributor_pr_count", contributor_pr_count)
print("contributor_pr_review_count", contributor_pr_review_count)
print("contributor_changedFiles_count", contributor_changedFiles_count)
print("contributor_changedLOC", contributor_changedLOC)
print("unattributed (pull requests):", unattributed["pr"])


In [ ]:
# This function processes the issue details of a repository, keyed by GitHub login
def issue_details(reponame):
    invalidate(
        [
            "contributor_issue_count",
            "contributor_comment_count",
            "df_issues",
            "df_issues_comments",
        ],
        "issues",
    )
    contributor_issue_count = dict()
    contributor_comment_count = dict()

    # loop though all the commit pages until the last returned empty page
    ipage = 1
    while True:
        spage = str(ipage)
        issue_api = "https://api.github.com/repos/" + reponame + "/issues?page=" + spage + "&per_page=100&state=all"
        issue_array = fetch(issue_api)

        # break out of the while loop if there are no more commits in the pages
        if len(issue_array) == 0:
            # print("no issues", issue_api)
            break

        # iterate through the list of issues and colloect the issue contributors
        for issue_obj in issue_array:

            created_time = pd.to_datetime(issue_obj["created_at"], utc=True)

            # Only count issues created strictly after the ignore_date
            if "pull_request" not in issue_obj.keys() and created_time > ignore_date:

                author = login_of(issue_obj["user"], "issues")

                # A deleted author's issue cannot be credited, but its commenters still are
                if author is not None:
                    # Original creator of the issue
                    contributor_issue_count[author] = contributor_issue_count.get(author, 0) + 1

                # Add the number of comments to the collection array
                issue_comments_api = issue_obj["comments_url"]
                issue_comment_array = fetch_all(issue_comments_api + "?per_page=100")
                for issue_comment_obj in issue_comment_array:
                    commenter = login_of(issue_comment_obj["user"], "issues")
                    if commenter is None:
                        continue
                    contributor_comment_count[commenter] = contributor_comment_count.get(commenter, 0) + 1

        ipage += 1
    return contributor_issue_count, contributor_comment_count


contributor_issue_count, contributor_comment_count = issue_details(reponame)

print("contributor_issue_count", contributor_issue_count)
print("contributor_comment_count", contributor_comment_count)
print("unattributed (issues):", unattributed["issues"])


In [ ]:
df_pr = pd.DataFrame(contributor_pr_count.items(), columns=["Login", "PRs"])
df_prReviews = pd.DataFrame(contributor_pr_review_count.items(), columns=["Login", "PR_Reviews"])
df_pr_ChangedFiles = pd.DataFrame(contributor_changedFiles_count.items(), columns=["Login", "Changed_Files"])
df_pr_ChangedLOC = pd.DataFrame(contributor_changedLOC.items(), columns=["Login", "Changed_LOC"])
df_issues = pd.DataFrame(contributor_issue_count.items(), columns=["Login", "Issues"])
df_issues_comments = pd.DataFrame(contributor_comment_count.items(), columns=["Login", "Issue_Comments"])


In [ ]:
from functools import reduce

df_list = [
    df_pr,
    df_prReviews,
    df_issues,
    df_issues_comments,
    df_pr_ChangedFiles,
    df_pr_ChangedLOC,
]
df_merged = reduce(lambda left, right: pd.merge(left, right, on=["Login"], how="outer"), df_list)
df_merged = df_merged.fillna(0)

# Every collected participant stays in the table. The metric columns are whole-number counts.
metric_columns = ["PRs", "PR_Reviews", "Issues", "Issue_Comments", "Changed_Files", "Changed_LOC"]
df_merged[metric_columns] = df_merged[metric_columns].astype(int)

# Sort by lowercase login, then by the login itself, so logins that differ only in case keep a fixed order
login_order = sorted(df_merged["Login"], key=lambda login: (login.lower(), login))
df_merged = df_merged.set_index("Login").loc[login_order].reset_index()

if df_merged.empty:
    print("No participants were collected, so the table and the charts are empty.")


**What the charts measure.** The file and line charts count activity in the pull requests each participant authored. Each merged PR's changed files are attributed to its author. Only files that pass the extension filter in `ignored_file_extensions` are counted, and a file changed in several PRs is counted once for each of them. These totals show PR-associated activity. They are not proof of who wrote the code, and they do not measure the quality of anyone's contribution.

In [ ]:
# Login is each participant's GitHub login

import matplotlib.pyplot as plt

# An empty table has nothing to plot, so this chart is skipped
if not df_merged.empty:
    fig = plt.figure()

    plt.style.use("classic")
    ax = df_merged.plot(x="Login", y=["PRs", "PR_Reviews", "Issues", "Issue_Comments"], kind="bar", rot=45)
    ax.set_xlabel("GitHub login")
    ax.set_ylabel("Count (PRs, PR reviews, issues, issue comments)")
    plt.tight_layout()
    plt.show()
# plt.savefig('../img/contributions.pdf')


In [ ]:
import matplotlib.pyplot as plt

if not df_merged.empty:
    plt.style.use("classic")
    ax = df_merged.plot(x="Login", y=["Changed_Files"], kind="bar", rot=45)
    ax.set_xlabel("GitHub login")
    ax.set_ylabel("File changes across authored PRs")
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

if not df_merged.empty:
    fig = plt.figure()

    plt.style.use("classic")
    ax = df_merged.plot(x="Login", y=["Changed_LOC"], kind="bar", rot=45)
    ax.set_xlabel("GitHub login")
    ax.set_ylabel("Lines added + deleted across authored PRs")
    plt.tight_layout()
    plt.show()
